## Travel Recommendation Agent using BeeAI Framework

## Basic Prompt Template for Travel Recommendations

In [ ]:
from pydantic import BaseModel, Field

from beeai_framework.template import PromptTemplate, PromptTemplateInput


# Define the input schema for our travel template
class TravelRecommendationInput(BaseModel):
    destination: str
    interests: str
    duration: str


# Create a basic travel recommendation template
travel_template: PromptTemplate = PromptTemplate(
    PromptTemplateInput(
        schema=TravelRecommendationInput,
        template="""
Destination: {{destination}}
Traveler's Interests: {{interests}}
Trip Duration: {{duration}}

Provide tailored travel recommendations for the destination above, focusing on the traveler's 
specific interests and considering the trip duration. Include suggestions for attractions,
activities, and local experiences.""",
    )
)

In [ ]:
# Render the template with sample data
prompt = travel_template.render(
    TravelRecommendationInput(
        destination="Kyoto, Japan",
        interests="Traditional culture, gardens, and food",
        duration="5 days",
    )
)

# Print the rendered prompt
print(prompt)

## advanced Template with Conditional Logic

In [ ]:
class TravelConstraint(BaseModel):
    type: str  # "budget", "accessibility", "season", etc.
    details: str


class AdvancedTravelInput(BaseModel):
    destination: str
    interests: str
    duration: str
    constraints: list[TravelConstraint]
    first_time_visitor: bool


# Create an advanced travel template with conditional sections
advanced_travel_template: PromptTemplate = PromptTemplate(
    PromptTemplateInput(
        schema=AdvancedTravelInput,
        template="""
Destination: {{destination}}
Traveler's Interests: {{interests}}
Trip Duration: {{duration}}

{{#first_time_visitor}}
Note: This is the traveler's first visit to this destination.
{{/first_time_visitor}}

{{#constraints.0}}
Important constraints to consider:
{{#constraints}}
- {{type}}: {{details}}
{{/constraints}}
{{/constraints.0}}

Provide tailored travel recommendations for the destination above, focusing on the traveler's 
specific interests and considering the trip duration. Include suggestions for attractions,
activities, and local experiences.
{{#first_time_visitor}}
Also include must-see attractions for first-time visitors.
{{/first_time_visitor}}""",
    )
)

In [ ]:
# Render the template with sample data
advanced_prompt = advanced_travel_template.render(
    AdvancedTravelInput(
        destination="Barcelona, Spain",
        interests="Architecture, beaches, and cuisine",
        duration="7 days",
        constraints=[
            TravelConstraint(type="Budget", details="Mid-range, approximately ?150/day"),
            TravelConstraint(type="Season", details="Summer (July-August)"),
        ],
        first_time_visitor=True,
    )
)

# Print the rendered prompt
print(advanced_prompt)

## gearing up ChatModel for Travel Recommendations

In [ ]:
from beeai_framework.backend.chat import ChatModel
from beeai_framework.backend.message import UserMessage, AssistantMessage, SystemMessage
from beeai_framework.backend.types import ChatModelOutput

# Set up credentials for WatsonX
WATSONX_API_KEY = ""
WATSONX_PROJECT_ID = ""
WATSONX_API_URL = "https://us-south.ml.cloud.ibm.com"


# Create a ChatModel instance
model = ChatModel.from_name(
    "watsonx:ibm/granite-3-8b-instruct",
    options={
        "project_id": WATSONX_PROJECT_ID,
        "api_key": WATSONX_API_KEY,
        "api_base": WATSONX_API_URL,
    },
)

In [ ]:
# Send a simple travel query to the model
user_message = UserMessage(content="What are the top 3 things to do in Tokyo for a first-time visitor?")

# Get the model's response
output: ChatModelOutput = await model.create(messages=[user_message])

print("Model Response:")
print(output.get_text_content())

## Building conversation memory

In [ ]:
from beeai_framework.memory.unconstrained_memory import UnconstrainedMemory

# Initialize memory
travel_memory = UnconstrainedMemory()

# Start a conversation about travel plans
await travel_memory.add_many(
    [
        UserMessage(content="I'm planning a trip to Thailand next month."),
    ]
)

# Get the model's response
first_response: ChatModelOutput = await model.create(messages=travel_memory.messages)
print("First Response:")
print(first_response.get_text_content())

# Add the assistant's response to memory and ask a follow-up question
await travel_memory.add_many(
    [
        AssistantMessage(content=first_response.get_text_content()),
        UserMessage(content="What should I pack for the weather there?"),
    ]
)

# Get the model's follow-up response
follow_up_response: ChatModelOutput = await model.create(messages=travel_memory.messages)
print("\nFollow-up Response:")
print(follow_up_response.get_text_content())

## structured travel recommendations

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field


class Activity(BaseModel):
    name: str = Field(description="Name of the activity or attraction")
    category: Literal["Cultural", "Nature", "Food", "Shopping", "Adventure"] = Field(
        description="Category of the activity"
    )
    estimated_time: str = Field(description="Estimated time needed for this activity")
    description: str = Field(description="Brief description of the activity")


class DayItinerary(BaseModel):
    day_number: int = Field(description="Day number in the itinerary")
    morning: Activity = Field(description="Morning activity")
    afternoon: Activity = Field(description="Afternoon activity")
    evening: Activity = Field(description="Evening activity or dinner recommendation")


class TravelItinerary(BaseModel):
    destination: str = Field(description="The travel destination")
    ideal_for: List[str] = Field(description="Types of travelers this itinerary is ideal for")
    days: List[DayItinerary] = Field(description="Day-by-day itinerary breakdown")
    estimated_budget: str = Field(description="Estimated budget range for this itinerary")
    travel_tips: List[str] = Field(description="Helpful tips specific to this destination")

In [ ]:
# Create a user message requesting an itinerary
itinerary_request = UserMessage(
    content="Please create a 3-day itinerary for Bali, Indonesia for a couple interested in both relaxation and adventure."
)

# Generate a structured travel itinerary
itinerary_response = await model.create_structure(schema=TravelItinerary, messages=[itinerary_request])

print("Structured Itinerary:")
print(itinerary_response.object)

In [ ]:
itinerary_response.object

## personalizing the travel agent with system prompts

In [ ]:
# Define different travel agent personalities
luxury_agent = SystemMessage(
    content="You are a luxury travel advisor specializing in high-end experiences. Your recommendations focus on exclusive, premium options and VIP experiences. Your tone is sophisticated and refined."
)

backpacker_agent = SystemMessage(
    content="You are a budget-conscious backpacker travel guide. You prioritize authentic local experiences, value for money, and off-the-beaten-path recommendations. Your tone is casual and adventurous."
)

family_agent = SystemMessage(
    content="You are a family travel specialist. You focus on child-friendly activities, safety, convenience, and experiences that appeal to multiple generations. Your tone is warm, helpful, and reassuring."
)

In [ ]:
# Test the luxury travel agent
luxury_query = UserMessage(content="Recommend a weekend in Paris.")
luxury_output: ChatModelOutput = await model.create(messages=[luxury_agent, luxury_query])

print("Luxury Travel Agent Response:")
print(luxury_output.get_text_content())

# Test the backpacker travel agent
backpacker_query = UserMessage(content="Recommend a weekend in Paris.")
backpacker_output: ChatModelOutput = await model.create(messages=[backpacker_agent, backpacker_query])

print("\nBackpacker Travel Agent Response:")
print(backpacker_output.get_text_content())

## putting all together: travel planning agent

In [ ]:

# Create a function to handle travel planning
async def travel_planning_agent(
    destination: str, 
    interests: str, 
    duration: str, 
    agent_type: str = "standard",
    previous_memory: UnconstrainedMemory = None
):
    # Initialize memory if not provided
    memory = previous_memory if previous_memory else UnconstrainedMemory()
    
    # Select the appropriate system prompt based on agent type
    if agent_type == "luxury":
        system = luxury_agent
    elif agent_type == "backpacker":
        system = backpacker_agent
    elif agent_type == "family":
        system = family_agent
    else:
        system = SystemMessage(content="You are a helpful travel planning assistant.")
    
    # If this is a new conversation, add the system message
    if not previous_memory:
        await memory.add(system)
    
    # Create a travel recommendation using our template
    travel_input = TravelRecommendationInput(
        destination=destination,
        interests=interests,
        duration=duration
    )
    
    prompt = travel_template.render(travel_input)
    
    # Add the templated question to memory
    await memory.add(UserMessage(content=prompt))
    
    # Get a response from the model
    response: ChatModelOutput = await model.create(messages=memory.messages)
    
    # Add the response to memory
    await memory.add(AssistantMessage(content=response.get_text_content()))
    
    return {
        "response": response.get_text_content(),
        "memory": memory
    }

In [ ]:
# Example usage of the travel planning agent
result = await travel_planning_agent(
    destination="New Zealand",
    interests="Hiking, nature photography, and local cuisine",
    duration="10 days",
    agent_type="backpacker"
)

print("Travel Planning Agent Response:")
print(result["response"])

# Continue the conversation with a follow-up question
memory = result["memory"]
await memory.add(UserMessage(content="What camera equipment would you recommend for this trip?"))

follow_up: ChatModelOutput = await model.create(messages=memory.messages)
print("\nFollow-up Response:")
print(follow_up.get_text_content())